# Practical Statistics for ML Interviews

> **Purpose:** A comprehensive, interview-ready reference covering statistical tests, feature selection metrics, and practical Python examples. Organized for quick review before data science interviews.

**What this notebook covers:**
1. **Reference Tables** -- Quick-lookup tables for statistical tests and feature selection metrics
2. **Interview Q&A: Statistical Tests** -- Detailed questions and answers on every major test
3. **Interview Q&A: Feature Selection Metrics** -- IV, WoE, MI, Cramer's V, VIF
4. **Practical Code Examples** -- Runnable Python demos for each concept
5. **EDA & Feature Engineering Pipeline** -- End-to-end pipeline Q&A
6. **Top Interview Questions** -- Rapid-fire one-liner answers for last-minute review

---

## 1. Reference Table: Statistical Tests

| Test                          | Data Scenario                                     | Purpose ("What it Measures")                                                                                         | When to Use                                                                                             | Assumptions                                                                                                                                          | Interpretation & Significance                                                                                                                                    | Strength (Power)                                                                                                                                            | Limitations / Best Scenario                                                                                            |
|-------------------------------|---------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------|
| **Shapiro-Wilk**              | Continuous, single sample                          | Tests normality of distribution                                                                                       | Before applying parametric tests (t-test, ANOVA)                                                        | Independent observations; continuous data                                                                                                            | H0: data are normally distributed. p<a => reject normality                                                                                                                  | High power for small to medium samples (n<2000)                                                                 | Sensitive to large n (always significant) -- use visual checks for large datasets                                         |
| **Levene's Test**             | Continuous, comparing variances across groups      | Tests equality of variances                                                                                            | Prior to t-test/ANOVA to check homoscedasticity                                                          | Independent observations; interval or ratio scale                                                                                                     | H0: variances equal. p<a => variances significantly differ                                                                                                              | Robust to non-normality                                                                                                  | If variances unequal, switch to Welch's variants                                                                       |
| **Student's t-Test**          | Two groups, continuous                             | Compares group means                                                                                                   | Binary target; balanced sample sizes; equal variances, approximate normality                             | Normality; equal variances; independent samples                                                                                                       | H0: mu1 = mu2. p<a => means differ                                                                                                                                            | High power under normality & equal variances                                                                            | Inflated Type I error if assumptions breached; use Welch's t-test instead                                              |
| **Welch's t-Test**            | Two groups, continuous, unequal variances/sizes    | Compares group means (allowing unequal variances)                                                                      | Binary target; imbalanced groups or unequal variances                                                   | Normality; independent samples                                                                                                                       | H0: mu1 = mu2. p<a => means differ                                                                                                                                            | More robust than Student's when variances/sizes differ                                                                 | Slightly lower power than Student's if variances truly equal                                                          |
| **Mann-Whitney U**            | Two groups, continuous/ordinal                     | Tests median difference; non-parametric                                                                                | Binary target; non-normal data or ordinal scales                                                         | Independent samples; similar distribution shapes (not strictly required)                                                                               | H0: distributions equal. p<a => medians differ                                                                                                                              | Good power for skewed data                                                                                               | Less power than t-test if normality holds; cannot quantify effect size easily                                          |
| **One-Way ANOVA**             | >=3 groups, continuous                              | Tests mean differences across multiple groups                                                                          | Multi-class target; balanced ANOVA assumptions met                                                       | Normality within groups; equal variances; independent observations                                                                                   | H0: all means equal. p<a => at least one mean differs                                                                                                                             | Higher power than multiple t-tests (controls Type I error)                                                              | Post-hoc tests needed to locate differences; sensitive to outliers                                                   |
| **Welch's ANOVA**             | >=3 groups, continuous, unequal variances/sizes     | Tests mean differences without assuming equal variances                                                                | Multi-class target; heteroscedasticity present                                                          | Normality; independent observations                                                                                                                 | H0: all means equal. p<a => at least one mean differs                                                                                                                             | Robust to unequal variances; better control of Type I error                                                            | Lower power than standard ANOVA if variances are equal                                                                 |
| **Kruskal-Wallis H-Test**     | >=3 groups, continuous/ordinal                      | Non-parametric ANOVA analogue (median differences)                                                                      | Multi-class target; non-normal or skewed data                                                            | Independent samples; ordinal or continuous data                                                                                                      | H0: group medians equal. p<a => at least one median differs                                                                                                                  | Robust to outliers & non-normality                                                                                      | Does not identify which groups differ; less power under normality                                                       |
| **Pearson Correlation**       | Continuous vs continuous                           | Measures linear association (r)                                                                                        | Checking linear relationships between two numeric features                                              | Bivariate normality; linearity; continuous; homoscedasticity                                                                                          | r near +/-1 indicates strong linear; test p<a => nonzero correlation                                                                                                        | High power for linear relationships                                                                                       | Misses non-linear associations; sensitive to outliers                                                                 |
| **Spearman Rank Correlation** | Continuous/ordinal vs continuous/ordinal           | Measures monotonic association                                                                                         | Non-linear or ordinal data, or when normality is violated                                               | Ordinal or continuous; monotonic relationship                                                                                                         | rho near +/-1 indicates strong monotonic; p<a => significant association                                                                                                       | Captures non-linear monotonic trends; robust to outliers                                                                 | Less sensitive to exact linearity; cannot capture non-monotonic relationships                                           |
| **Chi-Squared Test**          | Categorical vs categorical                         | Tests independence/association between two categorical variables                                                       | Categorical features vs categorical target; sufficient expected counts per cell (>=5)                     | Independent observations; large sample size                                                                                                          | H0: variables independent. p<a => association exists                                                                                                                          | Widely used; easy computation                                                                                             | Inaccurate if expected counts <5; use Fisher's exact in small samples                                                 |
| **Fisher's Exact Test**       | Categorical vs categorical (2x2 tables)            | Exact test for association in small samples                                                                            | 2x2 contingency, small expected counts (<5)                                                               | Independent observations                                                                                                                              | Calculates exact p-value; p<a => association exists                                                                                                                          | Accurate for small samples; exact p-value                                                                                  | Computationally intensive for large tables (>2x2)                                                                 |
| **Point-Biserial Correlation**| Continuous vs binary                              | Measures linear relationship between one continuous and one binary variable                                            | Continuous feature vs binary target during initial screening                                            | Continuous variable normally distributed within each binary group                                                                                   | r_pb near +/-1 indicates strong linear; p<a => significant association                                                                                                        | Translates easily from Pearson correlation                                                                                    | Sensitive to non-normality; only measures linear effects                                                                |
| **Kolmogorov-Smirnov Test**   | Two samples of continuous distributions            | Tests whether two samples come from the same distribution                                                              | Comparing feature distributions between churners vs non-churners                                         | Continuous data; independent samples                                                                                                                | H0: distributions identical. p<a => distributions differ                                                                                                                   | Non-parametric; sensitive to differences in both center and shape                                                           | Less powerful for specific shift types; sensitive to ties                                                               |

### Quick Decision Guide: Which Test to Pick?

```
Is your data CATEGORICAL?
  |-- Yes --> Are both variables categorical?
  |     |-- Yes --> Expected counts >= 5? --> Chi-Square
  |     |           Expected counts < 5?  --> Fisher's Exact
  |     |-- One categorical, one continuous (binary) --> Point-Biserial
  |
  |-- No (CONTINUOUS) --> How many groups?
        |-- 2 groups --> Normal + equal variance? --> Student's t-test
        |                Normal + unequal variance? --> Welch's t-test
        |                Non-normal? --> Mann-Whitney U
        |
        |-- 3+ groups --> Normal + equal variance? --> One-Way ANOVA
        |                 Normal + unequal variance? --> Welch's ANOVA
        |                 Non-normal? --> Kruskal-Wallis
        |
        |-- Correlation --> Linear + normal? --> Pearson
                           Monotonic / ordinal? --> Spearman
```

**Balanced vs. Imbalanced Usage Summary**
- **Balanced Groups (similar sizes):** Parametric tests (t-test, ANOVA) offer higher power when assumptions hold.
- **Imbalanced / Heteroscedastic Data:** Use Welch's variants or non-parametric alternatives (Mann-Whitney, Kruskal-Wallis) to maintain validity.
- **Categorical Features:** Chi-square for large samples; Fisher's exact for small cell counts.
- **Continuous Relationships:** Pearson for linear; Spearman for monotonic or non-normal data.
- **Distribution Checks:** Always validate normality (Shapiro-Wilk) and variance equality (Levene's) before selecting tests.

> **Interview Tip:** When asked "which test would you use?", always start by stating (1) what type the variables are, (2) how many groups, and (3) whether assumptions hold. This decision tree structure impresses interviewers far more than jumping to a test name.

## 2. Reference Table: Feature Selection Metrics

| Metric / Measure               | Data Scenario                                     | Purpose ("What it Measures")                                             | When to Use                                                                             | Assumptions                                                                                                                        | Interpretation & Significance                                                                                                                                                               | Strength (Power)                                                                                                                    | Limitations / Best Scenario                                                                                                                               |
|--------------------------------|---------------------------------------------------|---------------------------------------------------------------------------|-----------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Information Value (IV)**     | Categorical (including binned numerics) vs binary | Quantifies a feature's ability to separate two classes (good vs bad odds) | Pre-modeling filter for binary classification; credit scoring, churn prediction         | Requires bins for continuous features; non-zero event/non-event counts in each bin                                                    | IV < 0.02: uninterpretable; 0.02-0.1: weak; 0.1-0.3: medium; >0.3: strong predictor                                                                                                        | Directly tied to predictive power; robust for monotonic relationships                                                             | Sensitive to binning strategy; may overstate importance on small samples or poorly chosen bins                                                        |
| **Weight of Evidence (WoE)**    | Categorical vs binary                             | Encodes category's log-odds (ln[good%/bad%])                               | Feature engineering step preceding IV calculation; prepares continuous variables for logistic models | Same as IV (requires binning); no zero percentages in any bin                                                                      | WoE > 0: higher odds of "good" outcome; WoE < 0: higher odds of "bad" outcome; monotonic WoE indicates stable predictive power                                                            | Transforms categorical to continuous scale, easing model convergence                                                              | Not meaningful when target imbalance extreme; may suffer from overfitting if too many bins or rare categories                                          |
| **Mutual Information (MI)**     | Any feature type vs binary/continuous target      | Measures any dependency (linear or non-linear) between two variables       | Early filter to remove features with little to no dependency on target                      | No distributional assumptions                                                                                                       | MI ~ 0: independent; higher MI: stronger dependency. Absolute MI values not directly comparable across features -- rank relatively                                                            | Captures non-linear relationships; model-agnostic                                                                                  | Requires discretization for continuous-continuous or continuous-categorical; biased upward for high-cardinality features                                 |
| **Cramer's V**                  | Categorical vs categorical                        | Effect size measure of association strength (0 to 1)                      | Detect and remove redundant categorical features before encoding                            | Expected counts in contingency table should be reasonably large (>5)                                                               | V < 0.1: negligible; 0.1-0.3: weak; 0.3-0.5: moderate; >0.5: strong association. High V (e.g., >0.7) indicates near-duplication                                                                        | Single-number summary of association; symmetric for row/column categories                                                           | Inflated by large sample sizes; does not convey direction of association; not suited for ordinal data without adaptation                                |
| **Variance Inflation Factor (VIF)** | Continuous vs continuous                        | Quantifies multicollinearity: how much variance of a coefficient is inflated by other features | After numeric encoding, before modeling; to detect and remove highly collinear predictors    | Linear relationships among predictors; assumes features are numeric and approximately linear                                          | VIF = 1: no correlation; 1 < VIF < 5: moderate; VIF >= 5 (or >=10): high multicollinearity. Drop features with the highest VIF first in each correlated group                                | Directly targets multicollinearity, aiding linear-model stability                                                                  | Only applies to linear relationships; doesn't capture non-linear redundancy; requires model-fitting (e.g., regression) for each feature                     |

### One-Liner Cheat Sheet

| Metric | One-Liner |
|--------|-----------|
| **IV** | "How well does this feature separate classes?" (IV > 0.3 = strong) |
| **WoE** | "Which direction does each bin push the prediction?" (positive = good, negative = bad) |
| **MI** | "Is there ANY dependency between feature and target?" (captures non-linear) |
| **Cramer's V** | "How redundant are these two categorical features?" (V > 0.7 = near-duplicate) |
| **VIF** | "Is this feature a linear combo of others?" (VIF > 5 = multicollinear) |

**Usage Summary & Comparison**
- **Predictive Separation:** IV/WoE excels when you need direct measures of class separation, especially in binary classification with monotonic effects.
- **General Dependency:** MI is model-agnostic and captures any kind of dependency, ideal for initial broad filtering but needs careful discretization.
- **Categorical Redundancy:** Cramer's V pinpoints highly redundant categorical pairs, preventing unnecessary encoding burden.
- **Multicollinearity:** VIF is tailored to numeric features in linear contexts, ensuring model stability by removing correlated predictors.

> **Interview Tip:** When discussing feature selection, mention that you use a **layered approach**: start with IV/MI for relevance filtering, then Cramer's V for categorical redundancy, then VIF for numeric multicollinearity. This shows you understand that no single metric catches everything.

## 3. Interview Q&A: Statistical Tests

---

### Q1: What is a **Student's t-test**, and when would you use it?
- **Definition:** Compares the means of two groups (independent samples) to see if they differ significantly.
- **Assumptions:**
  1. Both groups are drawn from normally distributed populations.
  2. Variances are equal (for the standard "pooled" t-test).
  3. Observations are independent.
- **Use Case:** Comparing average churn rates between two customer segments.
- **Limitations:** Sensitive to outliers and non-normality. With skewed or ordinal data, it may be invalid.
- **One-liner:** "Compares two group means under normality + equal variance assumptions."

---

### Q2: What is the **Mann-Whitney U test**, and why choose it over a t-test?
- **Definition:** Non-parametric alternative to the t-test; compares distributions by ranking all observations.
- **Assumptions:**
  1. Observations are independent.
  2. The two samples have the same shape (though not normal).
- **Strengths:** Does not require normality. Robust to outliers.
- **When to Use:** Comparing median transaction values between churners and non-churners when the data are skewed.
- **Trade-off:** Tests for differences in distribution/median, not means. Less power than t-test when normality holds.
- **One-liner:** "Non-parametric rank test for two groups -- use when normality fails."

---

### Q3: What is **ANOVA**, and how does it differ from multiple t-tests?
- **Definition:** Analysis of Variance (ANOVA) tests whether three or more group means are equal.
- **Assumptions:**
  1. Normality of residuals.
  2. Homogeneity of variances.
  3. Independence.
- **Why Use Over Multiple t-tests:** Controls the overall Type I error rate. Single F-test vs. many pairwise comparisons.
- **Post-hoc:** If ANOVA is significant, use Tukey's HSD or Bonferroni to pinpoint which groups differ.
- **One-liner:** "Compares 3+ group means in one shot, controlling Type I error."

---

### Q4: What is the **Kruskal-Wallis** test, and when is it appropriate?
- **Definition:** Non-parametric equivalent of one-way ANOVA; compares medians across three or more groups.
- **Assumptions:**
  1. Independent observations.
  2. Similar distribution shapes.
- **Use Case:** Comparing median time-on-site across multiple customer tiers when data are non-normal.
- **Limitations:** Does not tell you which groups differ -- requires post-hoc pairwise tests (e.g., Dunn's test).
- **One-liner:** "Non-parametric ANOVA -- use for 3+ groups when normality fails."

---

### Q5: What is the **Chi-square test of independence**, and when would you apply it?
- **Definition:** Tests whether two categorical variables are related.
- **Assumptions:**
  1. Expected cell counts >= 5 (for validity).
  2. Independent observations.
- **Use Case:** Testing if churn status (yes/no) is independent of customer region (North/South/East/West).
- **Interpretation:** A significant p-value indicates an association but not its strength.
- **One-liner:** "Tests categorical independence -- significant p means association exists."

---

### Q6: How does **Cramer's V** build on Chi-square, and why use it?
- **Definition:** Standardized measure of association between two categorical variables, ranging from 0 (no association) to 1 (perfect).
- **Why Use:** Quantifies effect size, unlike Chi-square's p-value. Allows comparison across tables of different sizes.
- **Use Case:** Identifying and pruning highly redundant categorical features (e.g., two variables with V>0.8).
- **One-liner:** "Effect size for categorical association -- Chi-square tells IF, Cramer's V tells HOW MUCH."

---

### Q7: What is **Mutual Information (MI)**, and when is it helpful?
- **Definition:** Measures the reduction in uncertainty about one variable given knowledge of another; captures any dependency (linear or non-linear).
- **Strengths:** Model-agnostic relevance filter. Works for numeric and categorical data.
- **Use Case:** Ranking features by how much information they provide about churn, regardless of distribution shape.
- **Trade-off:** Does not indicate direction of relationship. Can be biased for highly cardinal features -- requires careful normalization or correction.
- **One-liner:** "Captures ANY dependency (even non-linear) between feature and target."

---

### Q8: When would you examine **Pearson vs. Spearman correlation**?
- **Pearson:** Measures linear correlation between two continuous, normally distributed variables.
- **Spearman:** Non-parametric; computes correlation on ranked data, capturing monotonic relationships.
- **Use Case:** Use Spearman when outliers or non-linear but monotonic relationships exist.
- **One-liner:** "Pearson = linear only. Spearman = any monotonic relationship."

---

### Quick Comparison Table

| Test                   | Variable Types               | Parametric? | Assumptions                  | Robust to Non-Normality | Effect Size Measure |
|------------------------|------------------------------|-------------|------------------------------|-------------------------|---------------------|
| t-test                 | 2 numeric groups             | Yes         | Normality, equal variances   | No                      | Cohen's d (optional)|
| Mann-Whitney U         | 2 ordinal/numeric groups     | No          | Independent, same shape      | Yes                     | --                  |
| ANOVA                  | >=3 numeric groups           | Yes         | Normality, homogeneity       | No                      | eta-squared, omega-squared |
| Kruskal-Wallis         | >=3 ordinal/numeric groups   | No          | Independent, same shape      | Yes                     | --                  |
| Chi-square             | 2+ categorical variables     | No (counts) | Expected counts, independence| N/A                     | --                  |
| Cramer's V             | 2 categorical variables      | --          | --                           | --                      | Gives effect size   |
| Mutual Information     | Numeric/categorical vs. target| No         | None                         | Yes                     | Value itself        |
| Pearson Correlation    | 2 continuous variables       | Yes         | Linearity, normality         | No                      | r                   |
| Spearman Correlation   | 2 ordinal/continuous         | No          | Monotonicity                 | Yes                     | rho                 |

---

**Summary (One-liner for Interview):**
Choosing the right statistical test hinges on your data's scale, distribution, and the question you're asking -- always match assumptions to reality.

**Key Takeaways:**
1. **Validate assumptions** (normality, variance, independence) before picking a test.
2. **Complement p-values with effect sizes** (Cohen's d, IV, Cramer's V, MI) to gauge practical significance.

> **Interview Tip:** If you can only remember one thing: *parametric tests assume normality and are more powerful when it holds; non-parametric tests are safer but less powerful.* Always mention that you check assumptions first before choosing.

## 4. Deep-Dive Interview Q&A: Tests & Feature Selection Metrics

This section probes deeper understanding -- the kind of follow-up questions interviewers ask when they want to see if you truly understand the "why" behind each method.

---

### 1. Normality Testing
**Q:** When and why would you use the Shapiro-Wilk test over visual methods (histogram, Q-Q plot)?
**A:** Use Shapiro-Wilk when you need a formal p-value for normality on a continuous feature before parametric testing. It is more sensitive than some tests for small/medium samples (n<2000). However, for very large datasets it will almost always reject normality -- so pair it with visual inspection.

---

### 2. Variance Homogeneity
**Q:** Why choose Levene's test instead of Bartlett's test for checking equal variances?
**A:** Bartlett's assumes normality; it can be misleading if data are even slightly non-normal. Levene's is robust to non-normal distributions, making it a safer first check before t-tests or ANOVA in real-world (often skewed) data.

---

### 3. Two-Group Mean Comparison
**Q:** What is the difference between Student's t-test and Welch's t-test, and how do you decide which to use?
**A:**
- **Student's** assumes equal variances; use when Levene's p>0.05 and group sizes are similar.
- **Welch's** relaxes that assumption; use if variances differ or sample sizes are unbalanced. It controls Type I error better under heteroscedasticity.

---

### 4. Non-Parametric Two-Group Test
**Q:** When is the Mann-Whitney U test preferred over any t-test?
**A:** When your continuous feature is heavily skewed or ordinal, and normality/variance assumptions fail. It compares group medians non-parametrically. Remember it tests distribution equality, not strictly medians.

---

### 5. Multi-Group Mean Comparison
**Q:** Compare One-Way ANOVA with Kruskal-Wallis. In what scenario is each best?
**A:**
- **ANOVA:** Use when each group's residuals are roughly normal and variances are equal. You get higher power and can follow up with Tukey's HSD for pairwise differences.
- **Kruskal-Wallis:** Use when those assumptions break down (skew, outliers, unequal variances) across 3+ groups. It is non-parametric, testing whether at least one group median differs.

---

### 6. Robust ANOVA Variant
**Q:** What is Welch's ANOVA, and why not always default to it?
**A:** Welch's ANOVA handles unequal variances across groups, like Welch's t-test for two groups. But if variances truly are equal, it has slightly less power than standard ANOVA. Use it only when Levene's indicates heteroscedasticity.

---

### 7. Correlation Measures
**Q:** When would you calculate Spearman's rho instead of Pearson's r?
**A:** When relationships are monotonic but not linear, or when features are ordinal/non-normal. Spearman uses rank-correlation, capturing monotonic trends and being robust to outliers.

---

### 8. Continuous-Binary Association
**Q:** What is the point-biserial correlation, and how does it differ from Pearson's r?
**A:** Point-biserial is a special case of Pearson's r where one variable is binary. It measures the linear association between a continuous predictor and a binary outcome, giving an interpretable r-value and p-value.

---

### 9. Categorical-Categorical Association
**Q:** How do you decide between Chi-square and Fisher's exact test?
**A:**
- **Chi-square:** Large sample sizes and expected cell counts >= 5. It is fast and scalable.
- **Fisher's:** Small samples or any 2x2 table with expected counts < 5. It gives an exact p-value but becomes computationally expensive for larger tables.

---

### 10. Distribution Comparison
**Q:** What insight does the Kolmogorov-Smirnov (K-S) test provide that Mann-Whitney does not?
**A:** K-S compares full cumulative distributions for two samples, detecting differences in shape, median, or variance. Mann-Whitney focuses on differences in central tendency (medians).

---

### 11. Predictive Power of Categoricals
**Q:** Why use Information Value (IV) and Weight of Evidence (WoE) instead of raw Chi-square p-values for feature selection?
**A:**
- **Chi-square p-value** only indicates if any association exists, not its strength or predictive stability.
- **IV/WoE** quantify the degree of separation (good vs. bad odds) and produce a monotonic encoding (WoE) ideal for logistic models. IV thresholds (e.g., >0.1 medium, >0.3 strong) directly map to predictive power.

---

### 12. General Dependency Filter
**Q:** How does Mutual Information (MI) complement IV and when might it mislead?
**A:** MI captures any dependency -- linear or non-linear -- between a feature and target, making it model-agnostic. However, MI is biased upward for high-cardinality features and requires careful binning or discretization to avoid spurious importance.

---

### 13. Categorical Redundancy
**Q:** What does Cramer's V tell you that Chi-square does not, and what is a safe cutoff for redundancy?
**A:** While Chi-square p-values depend on sample size, Cramer's V gives a normalized effect size (0-1) for association strength between two categoricals. A V > 0.7 suggests near-duplication, so you can safely drop one variable in a redundant pair.

---

### 14. Multicollinearity in Numerics
**Q:** Explain Variance Inflation Factor (VIF) and why it is preferred over raw correlation thresholds in some cases.
**A:** VIF measures how much a feature's variance is inflated by its linear relationships with *all* other features, not just pairwise. A VIF > 5 (or > 10) flags multicollinearity that can destabilize linear models. It is more holistic than a simple |corr| > 0.9 cutoff.

---

### 15. Choosing the Right Test -- Pipeline Example
**Q:** Outline the sequence of checks/tests you would run on a new numeric feature before modeling.
**A:**
1. **Missingness:** Drop or plan imputation if >90% null.
2. **Distribution:** Shapiro-Wilk + hist/Q-Q plot.
3. **Variance Equality (if comparing groups):** Levene's.
4. **Group Comparison (binary target):** If normal & equal var -> t-test/Welch; else -> Mann-Whitney.
5. **Multi-class comparison:** ANOVA/Welch ANOVA or Kruskal-Wallis.
6. **Correlation with other features:** Pearson/Spearman + VIF.
7. **Mutual information** with target as an overall relevance filter.

---

### 16. Advanced Pipeline Integration
**Q:** How do you blend univariate tests with model-based importance in your selection pipeline?
**A:** Start with broad filters (missingness thresholds, IV/MI), then univariate tests (Mann-Whitney, Chi-square) to remove clear non-contributors. Next, handle multicollinearity (VIF, Cramer's V), and finally run cross-validated tree-based models (e.g., LightGBM) to capture interactions -- keeping features with consistent positive importance scores.

> **Interview Tip:** Questions 15 and 16 are extremely common in senior DS interviews. Practice describing your end-to-end pipeline verbally in under 2 minutes. Interviewers want to hear a structured, sequential thought process -- not just test names.

## 5. Practical Code Examples

Runnable Python demonstrations of every key statistical test. Run this cell to see each test in action with synthetic data.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
import matplotlib.pyplot as plt

np.random.seed(0)

# ============================================================
# 1. NORMALITY CHECK + T-TEST (Two Normal Groups)
# ============================================================
group1 = np.random.normal(loc=0, scale=1, size=100)
group2 = np.random.normal(loc=0.5, scale=1, size=100)

shapiro1 = stats.shapiro(group1)
shapiro2 = stats.shapiro(group2)
t_stat, p_t = stats.ttest_ind(group1, group2)

print("=" * 60)
print("1. NORMALITY CHECK + T-TEST")
print("=" * 60)
print(f"Shapiro-Wilk p-values: group1 = {shapiro1.pvalue:.4f}, group2 = {shapiro2.pvalue:.4f}")
print(f"Student's t-test p-value: {p_t:.4f}")
print(f"Interpretation: {'Significant difference' if p_t < 0.05 else 'No significant difference'} between group means")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(group1, bins=20, color='steelblue', alpha=0.7)
axes[0].set_title("Group 1 (Normal, mean=0)")
axes[1].hist(group2, bins=20, color='coral', alpha=0.7)
axes[1].set_title("Group 2 (Normal, mean=0.5)")
plt.tight_layout()
plt.show()

# ============================================================
# 2. MANN-WHITNEY U (Skewed / Non-Normal Data)
# ============================================================
skew1 = np.random.exponential(scale=1, size=100)
skew2 = np.random.exponential(scale=1.5, size=100)

shapiro_skew1 = stats.shapiro(skew1)
shapiro_skew2 = stats.shapiro(skew2)
mw_stat, p_mw = stats.mannwhitneyu(skew1, skew2, alternative='two-sided')

print("\n" + "=" * 60)
print("2. MANN-WHITNEY U (Non-Normal Data)")
print("=" * 60)
print(f"Shapiro-Wilk p-values: skew1 = {shapiro_skew1.pvalue:.4f}, skew2 = {shapiro_skew2.pvalue:.4f}")
print(f"  -> Both non-normal (p < 0.05), so t-test is inappropriate")
print(f"Mann-Whitney U p-value: {p_mw:.4f}")
print(f"Interpretation: {'Significant' if p_mw < 0.05 else 'No significant'} distribution difference")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(skew1, bins=20, color='steelblue', alpha=0.7)
axes[0].set_title("Skew1 (Exponential, scale=1)")
axes[1].hist(skew2, bins=20, color='coral', alpha=0.7)
axes[1].set_title("Skew2 (Exponential, scale=1.5)")
plt.tight_layout()
plt.show()

# ============================================================
# 3. ANOVA vs. KRUSKAL-WALLIS (3+ Groups)
# ============================================================
g1 = np.random.normal(0, 1, 50)
g2 = np.random.normal(0.5, 1, 50)
g3 = np.random.normal(1, 1, 50)

f_stat, p_anova = stats.f_oneway(g1, g2, g3)
kw_stat, p_kw = stats.kruskal(g1, g2, g3)

print("\n" + "=" * 60)
print("3. ANOVA vs. KRUSKAL-WALLIS (3 Groups)")
print("=" * 60)
print(f"One-Way ANOVA: F={f_stat:.3f}, p={p_anova:.4f}")
print(f"Kruskal-Wallis: H={kw_stat:.3f}, p={p_kw:.4f}")
print(f"Interpretation: Both detect significant group differences")

plt.figure(figsize=(6, 3))
plt.boxplot([g1, g2, g3], labels=['G1 (mean=0)', 'G2 (mean=0.5)', 'G3 (mean=1)'])
plt.title("Boxplot: Three Normal Groups")
plt.tight_layout()
plt.show()

# ============================================================
# 4. CHI-SQUARE & CRAMER'S V (Categorical Association)
# ============================================================
cat1 = np.random.choice(['A', 'B', 'C'], size=200)
cat2 = np.random.choice(['X', 'Y'], size=200)
cont = pd.crosstab(cat1, cat2)

chi2, p_chi, dof, ex = stats.chi2_contingency(cont)
n = cont.values.sum()
phi2 = chi2 / n
r, k = cont.shape
cramer_v = np.sqrt(phi2 / min(r - 1, k - 1))

print("\n" + "=" * 60)
print("4. CHI-SQUARE & CRAMER'S V")
print("=" * 60)
print(f"Chi-square: chi2={chi2:.3f}, p={p_chi:.4f}")
print(f"Cramer's V: {cramer_v:.4f}")
print(f"Interpretation: {'Association exists' if p_chi < 0.05 else 'No association'} (V={cramer_v:.2f} = {'weak' if cramer_v < 0.1 else 'moderate' if cramer_v < 0.3 else 'strong'})")

plt.figure(figsize=(6, 3))
cont.plot(kind='bar')
plt.title("Contingency Counts: Cat1 vs Cat2")
plt.tight_layout()
plt.show()

# ============================================================
# 5. MUTUAL INFORMATION (Feature Relevance)
# ============================================================
df = pd.DataFrame({
    'normal_feat': np.concatenate([np.random.normal(0, 1, 100), np.random.normal(1, 1, 100)]),
    'skew_feat': np.concatenate([np.random.exponential(1, 100), np.random.exponential(1.5, 100)]),
    'uncorr': np.random.rand(200)
})
target = np.array([0] * 100 + [1] * 100)
mi = mutual_info_classif(df, target, discrete_features=[False, False, False], random_state=0)

print("\n" + "=" * 60)
print("5. MUTUAL INFORMATION")
print("=" * 60)
for feat, val in zip(df.columns, mi):
    signal = "useful predictor" if val > 0.01 else "weak/no signal"
    print(f"  MI({feat}, target) = {val:.4f}  -> {signal}")

# ============================================================
# 6. PEARSON vs. SPEARMAN CORRELATION
# ============================================================
x = np.linspace(0, 10, 100)
y_lin = 2 * x + np.random.normal(0, 1, 100)       # Linear
y_mon = x ** 2 + np.random.normal(0, 10, 100)      # Monotonic non-linear

r_p_lin, _ = stats.pearsonr(x, y_lin)
r_s_lin, _ = stats.spearmanr(x, y_lin)
r_p_mon, _ = stats.pearsonr(x, y_mon)
r_s_mon, _ = stats.spearmanr(x, y_mon)

print("\n" + "=" * 60)
print("6. PEARSON vs. SPEARMAN CORRELATION")
print("=" * 60)
print(f"Linear relationship:     Pearson r = {r_p_lin:.4f}, Spearman rho = {r_s_lin:.4f}")
print(f"Monotonic non-linear:    Pearson r = {r_p_mon:.4f}, Spearman rho = {r_s_mon:.4f}")
print(f"  -> Spearman captures the monotonic trend better (rho ~ r for linear, rho >= r for non-linear)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].scatter(x, y_lin, alpha=0.6, s=15)
axes[0].set_title(f"Linear: Pearson={r_p_lin:.2f}, Spearman={r_s_lin:.2f}")
axes[1].scatter(x, y_mon, alpha=0.6, s=15, color='coral')
axes[1].set_title(f"Monotonic: Pearson={r_p_mon:.2f}, Spearman={r_s_mon:.2f}")
plt.tight_layout()
plt.show()

# ============================================================
# 7. LEVENE'S TEST + WELCH'S T-TEST (Bonus: Unequal Variances)
# ============================================================
grp_a = np.random.normal(5, 1, 80)     # Small variance
grp_b = np.random.normal(5.5, 3, 120)  # Large variance, different size

lev_stat, p_lev = stats.levene(grp_a, grp_b)
welch_stat, p_welch = stats.ttest_ind(grp_a, grp_b, equal_var=False)

print("\n" + "=" * 60)
print("7. LEVENE'S TEST + WELCH'S T-TEST")
print("=" * 60)
print(f"Levene's test: stat={lev_stat:.3f}, p={p_lev:.4f}")
print(f"  -> {'Variances differ' if p_lev < 0.05 else 'Variances equal'} -> {'Use Welch' if p_lev < 0.05 else 'Student t-test OK'}")
print(f"Welch's t-test: t={welch_stat:.3f}, p={p_welch:.4f}")

# ============================================================
# 8. VIF CALCULATION (Multicollinearity Check)
# ============================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_df = pd.DataFrame({
    'x1': np.random.normal(0, 1, 200),
    'x2': np.random.normal(0, 1, 200),
})
vif_df['x3'] = vif_df['x1'] * 0.9 + np.random.normal(0, 0.3, 200)  # Highly correlated with x1
vif_df['intercept'] = 1

print("\n" + "=" * 60)
print("8. VIF (Variance Inflation Factor)")
print("=" * 60)
for i, col in enumerate(['x1', 'x2', 'x3']):
    vif = variance_inflation_factor(vif_df[['x1', 'x2', 'x3', 'intercept']].values, i)
    flag = " ** HIGH" if vif > 5 else ""
    print(f"  VIF({col}) = {vif:.2f}{flag}")
print("  -> x1 and x3 are collinear (VIF > 5); drop one before linear modeling")

### Code Results Summary & Key Takeaways

| Demo | Test Used | Result | Why This Test? |
|------|-----------|--------|----------------|
| **1. Normality + t-test** | Shapiro-Wilk, Student's t | p ~ 0.0004 (significant) | Normal data -> parametric test is valid and powerful |
| **2. Skewed data** | Shapiro-Wilk, Mann-Whitney U | Shapiro rejects normality; MW p ~ 0.0025 | Non-normal -> non-parametric test required |
| **3. Three groups** | ANOVA + Kruskal-Wallis | Both p ~ 0.0000 | Normal data -> ANOVA has more power; KW is the safe fallback |
| **4. Categorical** | Chi-square + Cramer's V | p ~ 0.58, V ~ 0.07 | No association; V confirms tiny effect size |
| **5. Feature relevance** | Mutual Information | normal_feat > skew_feat >> uncorr | MI captures any dependency; uncorr has zero signal |
| **6. Correlation** | Pearson vs Spearman | Both ~0.99 for linear; Spearman >= Pearson for monotonic | Spearman handles non-linear monotonic relationships |
| **7. Unequal variances** | Levene's + Welch's t | Levene's detects unequal var -> Welch's is appropriate | Always check variance equality before choosing t-test variant |
| **8. Multicollinearity** | VIF | x1 and x3 have VIF > 5 | VIF catches collinearity that pairwise correlation might miss |

**Key Decision Rules from the Code:**
- **Shapiro p < 0.05** -> normality violated -> switch to non-parametric tests
- **Levene's p < 0.05** -> variances unequal -> use Welch's variant
- **Chi-square p > 0.05** -> no categorical association; confirm with Cramer's V < 0.1
- **MI ~ 0** -> feature has no predictive value; safe to drop
- **VIF > 5** -> multicollinearity present; drop the less important feature

> **Interview Tip:** When discussing test results in an interview, always state three things: (1) the test you chose, (2) why you chose it over alternatives, and (3) the practical implication of the result. This shows methodical thinking.

## 6. Interview Q&A: EDA & Feature Engineering Pipeline

These questions test your ability to justify real-world pipeline decisions -- common in senior data science interviews.

---

### Q1: How did you decide which missing-value strategy to use (drop vs. impute vs. indicator)? Why not always impute?
**A:** Features with >90% missing carry minimal information -- dropping them prevents noise. For 50-90%, we imputed and added a missingness indicator to capture potential missing-value signal. Simple median/mode imputation for <10% preserves most data with minimal distortion. Always imputing without indicators loses the signal that "missingness" itself may correlate with the target.

---

### Q2: Why analyze unique-value counts before modeling? Can't an algorithm handle low/high cardinality features automatically?
**A:** Unique-value analysis informs encoding strategy: features with 2-3 uniques act like categorical -- LabelEncoding is simpler than continuous splitting. High-cardinality features (>10k) risk creating thousands of sparse dummy variables if one-hot encoded; preemptively planning for hashing or embeddings avoids explosion. Though tree models handle numeric splits, pre-defining categories often yields clearer feature engineering and interpretability.

---

### Q3: Why apply log-transform or binning for skewed distributions? Why not rely solely on tree-based models?
**A:** While tree ensembles are robust to monotonic transforms, binning helps highlight rare, extreme behaviors (e.g., top 1% spenders with 3x churn rate). Log-transform reduces skew to improve distance-based imputers and benefit algorithms like logistic regression or SVMs. Binning also simplifies interpretation and helps linear models capture non-linear effects.

---

### Q4: Why use Mann-Whitney U instead of t-test for numeric univariate filtering?
**A:** Our features were skewed and non-normal (mean skew=4.2), violating t-test and ANOVA assumptions of normality and equal variances. Mann-Whitney U is non-parametric, comparing medians robustly. ANOVA is also parametric, so inappropriate here.

---

### Q5: Why cross-verify univariate test results before dropping features? Why not rely solely on p-values?
**A:** P-values ignore interaction effects and can flag weak features as significant (multiple testing). By requiring an additional quality check -- high missingness, low variance post-imputation, or poor correlation with the target -- we avoid discarding features that might be important in multivariate contexts or interactions.

---

### Q6: For categorical features, why go beyond Chi-square p-values and compute Information Value (IV)?
**A:** Chi-square only tests dependence, not predictive power. IV quantifies a feature's ability to separate positive vs. negative classes, yielding a ranking. Features with IV<0.02 were removed as weak predictors, which Chi-square alone might not catch if sample size is large enough to yield "significance."

---

### Q7: How does Cramer's V differ from Chi-square, and why use it?
**A:** Cramer's V measures effect size (association strength) between two categorical variables, normalized between 0 and 1. Chi-square's p-value indicates significance but not magnitude. We used Cramer's V>0.7 to identify redundant categorical pairs and dropped the lower-IV member to reduce multicollinearity among categoricals.

---

### Q8: Explain the iterative correlation removal process. Why not use PCA to handle multicollinearity?
**A:** PCA transforms features into orthogonal components, losing interpretability. Our iterative removal preserves original features, dropping only the less predictive or noisier variable in each highly correlated pair (>0.9). This maintains feature meaning -- crucial for stakeholder buy-in -- while reducing redundancy.

---

### Q9: Why perform Kruskal-Wallis interaction tests? Could interaction terms in modeling suffice?
**A:** Kruskal-Wallis tests reveal which numeric-categorical pairs have statistically different distributions, highlighting "hub" features that drive interactions. While tree models capture interactions automatically, knowing these hubs informs feature engineering (e.g., manual interaction terms) and supports interpretability and hypothesis-driven exploration.

---

### Q10: Why include Mutual Information (MI) after interaction analysis?
**A:** MI detects any dependency -- linear or non-linear -- between feature and target, serving as a broad relevance filter. While earlier steps ensure quality and reduce redundancy, MI quickly removes features with near-zero dependency, further refining the set before model-based importance.

---

### Q11: How did you handle class imbalance during modeling? Why not always use SMOTE or downsampling?
**A:** We used scale_pos_weight in LightGBM and experimented with SMOTE on training folds. SMOTE can create synthetic minority samples but may introduce noise and lead to overfitting if not carefully tuned. Using model weights preserves original data distribution and mitigates overfitting risks.

---

### Q12: How would you incorporate high-cardinality features in a deep learning model?
**A:** One-hot encoding >10k categories is infeasible. In deep models, embedding layers map each category to a low-dimensional vector learned during training. Alternatively, frequency or target encoding reduces dimensionality. Hashing trick is another option, trading off collisions for fixed dimension.

---

### Q13: What are the risks of data leakage in this pipeline, and how did you mitigate them?
**A:** Data leakage can occur via improper imputation (using test-set stats), target encoding without cross-validation, or selecting features based on test data. We performed all calculations (imputation stats, MI, importance) within cross-validation folds on training data only. Missingness indicators, IV, and target encoding were computed in a CV-aware manner to prevent leakage.

> **Interview Tip:** Data leakage is one of the most common follow-up questions in ML interviews. Always proactively mention that you compute all statistics (imputation, encoding, feature selection) on training data only, using cross-validation folds. This immediately signals experience with production pipelines.

## 7. End-to-End EDA & Feature Selection Pipeline

This table serves as a step-by-step checklist you can reference during interviews when asked "Walk me through your feature engineering pipeline."

| Step | What & Why | Tools / Tests | Expected Outcome & Effect |
|:----:|:-----------|:--------------|:--------------------------|
| **1. Data Audit & Load** | Inspect raw shape, dtypes, memory. Catch glaring issues (empty columns, wrong types). | `df.info()`, `df.memory_usage()` | Baseline snapshot: rows/cols/types/memory, guiding next optimizations. |
| **2. Data-Type & Memory Optimization** | Down-cast numerics, convert low-card `object` to `category`. Speeds up all subsequent ops. | `pd.to_numeric()`, `df.astype('category')` | Reduced footprint; clear numeric vs. categorical split. |
| **3. Missing-Value Profiling** | Quantify per-feature missingness. Set drop vs. impute vs. flag thresholds. | `df.isna().mean()`, heatmap | >90% drop; 50-90% impute+flag; <10% simple impute. |
| **4. Cardinality & Uniqueness** | Identify binary/low-card feats for simple encoding. Spot high-card for special treatment. | `df.nunique()`, value-count sampling | Low (<=5) categorical flags; high (>10k) consider hashing/target-encode. |
| **5. Distribution & Outlier Checks** | Measure skew/kurtosis; plot hist/box. Decide transforms or binning. | `df.skew()`, `sns.histplot()`, IQR/Z-score | Skew >1 log1p/Box-Cox or Low/Med/High bins; outliers identified for capping. |
| **6. Normality & Variance Tests** | Choose parametric vs. non-parametric tests. | Shapiro-Wilk, Levene's | p<0.05 on Shapiro non-normal; Levene's p<0.05 heteroscedastic use Welch or non-param tests. |
| **7. Univariate Feature-Target Tests** | Initial relevance filter. | Binary: Mann-Whitney U / Chi-square / Fisher's. Multi-class: ANOVA / Welch ANOVA / Kruskal-Wallis | p<0.05 feature shows significant group separation. |
| **8. Cross-Verify "Insignificant"** | Don't drop on p-value alone. Check other red flags. | Combine Step 3 & low variance & low corr | Drop only if p>0.05 AND (high missingness OR near-zero variance OR negligible corr). |
| **9. Categorical Predictiveness & Redundancy** | Measure class-separation power and remove dupes. | IV/WoE, Cramer's V | IV<0.02 drop; Cramer's V>0.7 drop lower-IV of pair. |
| **10. Numeric Multicollinearity** | Remove redundant numerics. | Pearson/Spearman corr, VIF or iterative removal | Iteratively drop |corr|>0.9 pairs -- keep feature with higher target correlation. |
| **11. Interaction Analysis** | Uncover key num-cat interactions. | Kruskal-Wallis, ANOVA variants | Identify "hub" features in many significant interactions -- protect them from drop. |
| **12. Model-Based Relevance** | Capture non-linear & interaction effects. | MI filter, CV-LightGBM importance | Keep features with MI>threshold and mean LGBM importance>0. |
| **13. Final Correlation Cleanup** | Ensure no residual numeric redundancy. | Repeat Step 10 on final set | Zero out remaining high-corr pairs. |
| **14. Feature Engineering & Encoding** | Impute, encode, scale, flag missings. | Median/mode/KNN imputer; One-hot / ordinal / target encoding; Scaler | Ready-to-model dataset. Missingness flags retained if predictive. |
| **15. Evaluation Setup** | Handle imbalance/multiclass; choose metrics. | Stratified K-Fold; Metrics: AUC-PR (binary), macro-F1 (multi), confusion matrix | Robust, fair performance estimates; metrics aligned to business cost. |
| **16. Complex Data Scenarios** | Time-Series: lag/rolling features, temporal splits. Hierarchical: group-wise imputers, GroupKFold. Text: TF-IDF/embeddings. Multi-Modal: separate pipelines, late fusion. | statsmodels, sklearn.compose.ColumnTransformer, HuggingFace Transformers | Pipeline extensions that respect each data type's structure -- no leakage, valid features. |

---

**Key Takeaways for Complex Data:**

- **Time-Aware Splitting**: Always split train/test along temporal boundaries to prevent leakage.
- **Group Integrity**: For hierarchical or panel data, use GroupKFold or mixed-models to honor within-group correlations.
- **Text/Graph**: Perform dedicated EDA (word counts, degree distributions), then choose embeddings that capture semantics or structure.
- **Imputation with Context**: Leverage domain patterns -- e.g., forward-fill in time series, group-median for panel data.
- **End-to-End Pipelines**: Encapsulate modality-specific transformations in `ColumnTransformer` or custom pipeline steps to maintain reproducibility.

> **Interview Tip:** Memorize this 16-step pipeline. When asked "walk me through your approach to a new dataset," recite the high-level steps (audit, missing values, distributions, tests, multicollinearity, model-based filtering, encoding, evaluation). You do not need every detail -- the structure alone demonstrates seniority.

## 8. Top Interview Questions: Rapid-Fire One-Liners

Use this section for last-minute review. Each answer is designed to be spoken in under 15 seconds.

---

### Statistical Tests

| # | Question | One-Liner Answer |
|---|----------|-----------------|
| 1 | What is a p-value? | The probability of observing data this extreme if the null hypothesis were true. p < 0.05 means statistically significant. |
| 2 | Type I vs Type II error? | Type I = false positive (reject true H0). Type II = false negative (fail to reject false H0). |
| 3 | When to use t-test vs Mann-Whitney? | t-test for normal data with equal variances; Mann-Whitney when normality or equal variance fails. |
| 4 | When to use ANOVA vs Kruskal-Wallis? | ANOVA for 3+ normal groups with equal variances; Kruskal-Wallis when those assumptions fail. |
| 5 | What does Shapiro-Wilk test? | Whether a sample comes from a normal distribution. p < 0.05 means non-normal. |
| 6 | What does Levene's test check? | Equality of variances across groups. If p < 0.05, use Welch's variant. |
| 7 | Welch's vs Student's t-test? | Welch's does not assume equal variances -- it is safer and nearly as powerful. Default to Welch's. |
| 8 | Chi-square vs Fisher's exact? | Chi-square for large samples (expected counts >= 5); Fisher's for small samples or 2x2 tables. |
| 9 | Pearson vs Spearman? | Pearson measures linear correlation; Spearman measures any monotonic relationship using ranks. |
| 10 | What is the K-S test? | Compares two full distributions -- detects differences in shape, center, and spread. |
| 11 | What is point-biserial correlation? | Special case of Pearson's r where one variable is binary. Measures linear association with a binary target. |
| 12 | What is statistical power? | The probability of correctly rejecting a false H0. Increases with sample size and effect size. |
| 13 | What is the Bonferroni correction? | Divides alpha by the number of tests to control family-wise Type I error in multiple comparisons. |
| 14 | What is effect size and why does it matter? | Quantifies the magnitude of a difference (e.g., Cohen's d). p-values only say IF significant; effect size says HOW MUCH. |

### Feature Selection Metrics

| # | Question | One-Liner Answer |
|---|----------|-----------------|
| 15 | What is Information Value (IV)? | Measures how well a feature separates two classes. IV > 0.3 = strong predictor. |
| 16 | What is Weight of Evidence (WoE)? | Log-odds encoding of categories; WoE > 0 means favorable class, WoE < 0 means unfavorable. |
| 17 | What is Mutual Information? | Measures any dependency (including non-linear) between feature and target. MI = 0 means independent. |
| 18 | What is Cramer's V? | Normalized effect size (0-1) for association between two categorical variables. V > 0.7 = redundant pair. |
| 19 | What is VIF? | Measures how much a coefficient's variance is inflated by collinearity with ALL other features. VIF > 5 = problem. |
| 20 | IV vs MI -- when to use which? | IV for binary targets with monotonic separation (credit scoring). MI for any target type and non-linear dependencies. |

### Pipeline & Practical

| # | Question | One-Liner Answer |
|---|----------|-----------------|
| 21 | How do you check for data leakage? | Ensure all transformations (imputation, encoding, feature selection) are fitted on training data only. |
| 22 | How do you handle missing values? | >90% missing: drop. 50-90%: impute + add indicator flag. <10%: simple median/mode impute. |
| 23 | How do you handle class imbalance? | Model weights (scale_pos_weight), stratified sampling, SMOTE on training folds only, or AUC-PR as metric. |
| 24 | PCA vs iterative correlation removal? | PCA loses interpretability. Iterative removal preserves original features -- better for stakeholder communication. |
| 25 | Why not rely only on p-values for feature selection? | p-values inflate with large n, miss interactions, and don't measure effect size. Always pair with IV, MI, or model importance. |
| 26 | How do you handle high-cardinality categoricals? | Target encoding, frequency encoding, hashing trick, or embedding layers in deep models. Never one-hot encode >100 categories. |
| 27 | What is your feature selection pipeline order? | Missingness -> distributions -> univariate tests -> multicollinearity (VIF/Cramer's V) -> MI filter -> model-based importance. |
| 28 | What is the difference between statistical significance and practical significance? | Statistical significance (p < 0.05) means the effect is unlikely due to chance. Practical significance means the effect is large enough to matter in the real world. |
| 29 | When would you use Welch's ANOVA over standard ANOVA? | When Levene's test shows unequal variances across groups. Welch's is more robust but slightly less powerful. |
| 30 | How do you validate normality for large datasets (n > 2000)? | Shapiro-Wilk becomes too sensitive at large n -- use visual checks (Q-Q plot, histogram) alongside the formal test. |

---

> **Final Interview Tip:** In a statistics interview, the three most impressive things you can do are: (1) state assumptions before naming a test, (2) distinguish between statistical and practical significance, and (3) describe your complete pipeline as a sequence of principled decisions rather than isolated techniques.